#  Capa Oro: indicadores semanales y predicción

Lee de **silver** y escribe en **gold**:

| Tabla | Qué tiene |
|---|---|
| `gold.precio_semanal` | promedio, mínimo y máximo por semana (martes a lunes) y combustible |
| `gold.variacion_semanal` | diferencia Q y % contra la semana anterior y etiqueta `direccion` |
| `gold.features_modelo` | variables para predecir la dirección de la semana siguiente |
| `gold.predicciones` | predicción, probabilidad y (cuando se conoce) dirección real |
| `gold.metricas_modelo` | exactitud del modelo y de la línea base (para `/modelo/metricas`) |

**Fuente de cada semana:** se usan los precios leídos de las fotos (solo lecturas reales: `foto`, `foto_sin_referencia`,
`corregido_manual`; no los rellenados con MAGA). Las semanas sin fotos se completan con el promedio mensual del MAGA
(interpolado a semanas) y, si tampoco hay MAGA, se arrastra el último precio conocido. La columna `fuente` lo indica siempre.

Todo se recalcula desde silver y se escribe con `overwrite`, así que correrlo dos veces da el mismo resultado.

In [0]:
%pip install -q "scikit-learn>=1.3" "mlflow>=2.10"

In [0]:
%restart_python

In [0]:
# Parámetros (widgets): en el Job se pasan como parámetros de la tarea
def _widget(nombre, defecto, etiqueta, opciones=None):
    try:
        if opciones:
            dbutils.widgets.dropdown(nombre, defecto, opciones, etiqueta)
        else:
            dbutils.widgets.text(nombre, defecto, etiqueta)
        return dbutils.widgets.get(nombre)
    except NameError:                       # fuera de Databricks (pruebas locales)
        return defecto

CATALOGO        = _widget("catalogo", "equipo4", "Catálogo")
UMBRAL_Q        = float(_widget("umbral_direccion", "0.10", "Umbral sube/baja (Q)"))
SEMANAS_PRUEBA  = int(_widget("semanas_prueba", "52", "Semanas para prueba (las más recientes)"))
RELLENO_MAGA    = _widget("relleno_maga", "interpolado", "Semanas sin fotos", ["interpolado", "escalon", "ninguno"])
EXPERIMENTO     = _widget("experimento_mlflow", "", "Experimento MLflow (vacío = /Users/<tú>/<catálogo>-gasolina)")

OFICIALES       = ["diesel", "regular", "super"]
ORIGENES_FOTO   = ["foto", "foto_sin_referencia", "corregido_manual"]   # lecturas reales (no rellenadas)
CLASES          = ["baja", "se_mantiene", "sube"]

T_SILVER_LECT   = f"{CATALOGO}.silver.lecturas_fotos"
T_SILVER_OFIC   = f"{CATALOGO}.silver.precios_oficiales_mensuales"
T_PRECIO        = f"{CATALOGO}.gold.precio_semanal"
T_VARIACION     = f"{CATALOGO}.gold.variacion_semanal"
T_FEATURES      = f"{CATALOGO}.gold.features_modelo"
T_PREDICCIONES  = f"{CATALOGO}.gold.predicciones"
T_METRICAS      = f"{CATALOGO}.gold.metricas_modelo"

print(f"catálogo={CATALOGO}  umbral=±Q{UMBRAL_Q:.2f}  semanas_prueba={SEMANAS_PRUEBA}  relleno={RELLENO_MAGA}")

In [0]:
import numpy as np
import pandas as pd

def inicio_semana(fechas):
    """Martes de la semana (martes a lunes, como los precios de referencia del MEM)."""
    f = pd.to_datetime(fechas).dt.normalize()
    return f - pd.to_timedelta((f.dt.dayofweek - 1) % 7, unit="D")   # pandas: lunes=0, martes=1

def fotos_por_semana(lect):
    """Lecturas reales de las fotos -> promedio, mínimo y máximo por semana y combustible."""
    partes = []
    for c in OFICIALES:
        if c not in lect.columns:
            continue
        d = lect[["archivo", "fecha_hora", c, f"{c}_origen"]].rename(columns={c: "precio", f"{c}_origen": "origen"})
        d = d[d["origen"].isin(ORIGENES_FOTO) & d["precio"].notna() & d["fecha_hora"].notna()].copy()
        d["combustible"] = c
        partes.append(d)
    if not partes or all(p.empty for p in partes):
        return pd.DataFrame(columns=["combustible", "semana_inicio", "promedio", "minimo", "maximo", "n_lecturas"])
    d = pd.concat(partes, ignore_index=True)
    d["semana_inicio"] = inicio_semana(d["fecha_hora"])
    return (d.groupby(["combustible", "semana_inicio"])
             .agg(promedio=("precio", "mean"), minimo=("precio", "min"), maximo=("precio", "max"),
                  n_lecturas=("precio", "size"))
             .reset_index())

def maga_por_semana(ofi, semanas, modo):
    """Precio MAGA para cada semana. 'interpolado': línea entre los promedios mensuales (puestos a mitad de mes);
    'escalon': el promedio del mes en que cae la semana. Fuera del rango del MAGA queda vacío."""
    if modo == "ninguno" or ofi.empty:
        return pd.DataFrame(columns=["combustible", "semana_inicio", "precio_maga"])
    medio_semana = semanas + pd.Timedelta(days=3)
    filas = []
    for c in OFICIALES:
        s = ofi[ofi["combustible"] == c].sort_values("mes_inicio")
        if s.empty:
            continue
        ini, fin = s["mes_inicio"].min(), s["mes_inicio"].max() + pd.offsets.MonthEnd(0)
        dentro = (medio_semana >= ini) & (medio_semana <= fin)
        if modo == "interpolado":
            x = (s["mes_inicio"] + pd.Timedelta(days=14)).astype("int64").to_numpy()
            precio = np.interp(medio_semana.astype("int64").to_numpy(), x, s["precio_oficial"].to_numpy())
        else:
            por_mes = s.set_index(s["mes_inicio"].dt.to_period("M"))["precio_oficial"]
            precio = medio_semana.dt.to_period("M").map(por_mes).to_numpy(dtype=float)
        filas.append(pd.DataFrame({"combustible": c, "semana_inicio": semanas.to_numpy(),
                                   "precio_maga": np.where(dentro, precio, np.nan)}))
    return pd.concat(filas, ignore_index=True).dropna(subset=["precio_maga"])

def construir_precio_semanal(lect, ofi, modo=RELLENO_MAGA):
    fotos = fotos_por_semana(lect)
    inicios = [fotos["semana_inicio"].min()] if len(fotos) else []
    finales = [fotos["semana_inicio"].max()] if len(fotos) else []
    if modo != "ninguno" and len(ofi):
        inicios.append(inicio_semana(pd.Series([ofi["mes_inicio"].min()])).iloc[0])
        finales.append(inicio_semana(pd.Series([ofi["mes_inicio"].max() + pd.offsets.MonthEnd(0)])).iloc[0])
    if not inicios:
        raise ValueError("No hay datos en silver: corre primero los notebooks de bronze y silver")
    semanas = pd.Series(pd.date_range(min(inicios), max(finales), freq="7D"))   # calendario completo de martes
    maga = maga_por_semana(ofi, semanas, modo)

    base = pd.MultiIndex.from_product([OFICIALES, semanas], names=["combustible", "semana_inicio"]).to_frame(index=False)
    d = (base.merge(fotos, on=["combustible", "semana_inicio"], how="left")
             .merge(maga, on=["combustible", "semana_inicio"], how="left")
             .sort_values(["combustible", "semana_inicio"]))
    hay_foto = d["promedio"].notna()
    d["fuente"] = np.where(hay_foto, "foto", np.where(d["precio_maga"].notna(), f"maga_{modo}", None))
    for col in ("promedio", "minimo", "maximo"):
        d[col] = d[col].where(hay_foto, d["precio_maga"])
    # Semanas sin foto ni MAGA: se arrastra el último precio conocido de ese combustible
    sin_dato = d["promedio"].isna()
    d[["promedio", "minimo", "maximo"]] = d.groupby("combustible")[["promedio", "minimo", "maximo"]].ffill()
    d.loc[sin_dato & d["promedio"].notna(), "fuente"] = "arrastre"
    d = d[d["promedio"].notna()].copy()
    d["n_lecturas"] = d["n_lecturas"].fillna(0).astype(int)
    d["semana_fin"] = d["semana_inicio"] + pd.Timedelta(days=6)
    for col in ("promedio", "minimo", "maximo"):
        d[col] = d[col].round(2)
    return d[["combustible", "semana_inicio", "semana_fin", "promedio", "minimo", "maximo", "n_lecturas", "fuente"]] \
             .reset_index(drop=True)

def direccion(dif, umbral=UMBRAL_Q):
    return pd.Series(np.select([dif > umbral, dif < -umbral], ["sube", "baja"], "se_mantiene"),
                     index=dif.index).where(dif.notna())

def construir_variacion(precio):
    d = precio.sort_values(["combustible", "semana_inicio"]).copy()
    g = d.groupby("combustible")
    d["precio_anterior"] = g["promedio"].shift(1)
    d["fuente_anterior"] = g["fuente"].shift(1)
    d["dif_q"] = (d["promedio"] - d["precio_anterior"]).round(2)
    d["dif_pct"] = (100 * d["dif_q"] / d["precio_anterior"]).round(2)
    d["direccion"] = direccion(d["dif_q"])
    d["umbral_q"] = UMBRAL_Q
    return d[["combustible", "semana_inicio", "promedio", "precio_anterior", "dif_q", "dif_pct", "direccion",
              "umbral_q", "fuente", "fuente_anterior"]].reset_index(drop=True)

FEATURES = ["dif_1", "dif_2", "dif_3", "dif_4", "dif_pct_1", "desvio_media_4", "dif_acum_4",
            "spread_super_diesel", "spread_regular_diesel", "dir_actual",
            "es_diesel", "es_regular", "es_super"]

def construir_features(variacion):
    d = variacion.sort_values(["combustible", "semana_inicio"]).copy()
    g = d.groupby("combustible")
    d["dif_1"] = d["dif_q"]
    for k in (2, 3, 4):
        d[f"dif_{k}"] = g["dif_q"].shift(k - 1)                 # cambio de hace k-1 semanas
    d["dif_pct_1"] = d["dif_pct"]
    d["media_movil_4"] = g["promedio"].transform(lambda s: s.rolling(4, min_periods=4).mean()).round(4)
    d["desvio_media_4"] = (d["promedio"] - d["media_movil_4"]).round(4)
    d["dif_acum_4"] = (d["promedio"] - g["promedio"].shift(4)).round(4)
    ancho = d.pivot(index="semana_inicio", columns="combustible", values="promedio")
    spreads = pd.DataFrame({"semana_inicio": ancho.index,
                            "spread_super_diesel": (ancho.get("super") - ancho.get("diesel")).to_numpy(),
                            "spread_regular_diesel": (ancho.get("regular") - ancho.get("diesel")).to_numpy()})
    d = d.merge(spreads, on="semana_inicio", how="left")
    d["dir_actual"] = d["direccion"].map({"baja": -1.0, "se_mantiene": 0.0, "sube": 1.0})
    for c in OFICIALES:
        d[f"es_{c}"] = (d["combustible"] == c).astype(float)
    g = d.groupby("combustible")
    d["direccion_siguiente"] = g["direccion"].shift(-1)
    d["fuente_siguiente"] = g["fuente"].shift(-1)
    # Solo se entrena/evalúa con pares de semanas observadas (foto o MAGA), no con precios arrastrados
    d["apta_modelo"] = ((d["fuente"] != "arrastre") & (d["fuente_anterior"] != "arrastre")
                        & (d["fuente_siguiente"] != "arrastre") & d["direccion_siguiente"].notna()
                        & d[FEATURES].notna().all(axis=1))
    return d[["combustible", "semana_inicio", "promedio", *FEATURES, "media_movil_4", "fuente",
              "direccion", "direccion_siguiente", "apta_modelo"]].reset_index(drop=True)

def separar_temporal(feats, semanas_prueba=SEMANAS_PRUEBA):
    """Prueba = las últimas N semanas con etiqueta; entrenamiento = todo lo anterior (nunca aleatorio)."""
    etiq = feats[feats["apta_modelo"]]
    semanas = np.sort(etiq["semana_inicio"].unique())
    if len(semanas) < 10:
        raise ValueError(f"Solo hay {len(semanas)} semanas con etiqueta: muy pocas para entrenar")
    n = min(semanas_prueba, max(1, int(len(semanas) * 0.3)))   # nunca más del 30 % para prueba
    corte = semanas[-n]
    return etiq[etiq["semana_inicio"] < corte], etiq[etiq["semana_inicio"] >= corte], pd.Timestamp(corte)

## 2. Precio semanal, variación y variables

In [0]:
from pyspark.sql import functions as F

for t in (T_SILVER_LECT, T_SILVER_OFIC):
    if not spark.catalog.tableExists(t):
        raise RuntimeError(f"Falta {t}: corre primero el notebook de silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.gold")

columnas_lect = ["archivo", "fecha_hora"] + [x for c in OFICIALES for x in (c, f"{c}_origen")]
lect_pd = spark.table(T_SILVER_LECT).select(*[c for c in columnas_lect if c in spark.table(T_SILVER_LECT).columns]).toPandas()
ofi_pd  = spark.table(T_SILVER_OFIC).select("combustible", "mes_inicio", "precio_oficial").toPandas()
lect_pd["fecha_hora"] = pd.to_datetime(lect_pd["fecha_hora"])
ofi_pd["mes_inicio"]  = pd.to_datetime(ofi_pd["mes_inicio"])
print(f"silver: {len(lect_pd)} fotos, {len(ofi_pd)} precios mensuales MAGA")

precio_semanal    = construir_precio_semanal(lect_pd, ofi_pd)
variacion_semanal = construir_variacion(precio_semanal)
features_modelo   = construir_features(variacion_semanal)

print(precio_semanal.groupby(["combustible", "fuente"]).size().unstack(fill_value=0))

In [0]:
def guardar(pdf, tabla):
    """pandas -> Delta (overwrite: idempotente). Fechas como DATE y NaN como NULL."""
    pdf = pdf.copy()
    for c in pdf.columns:
        if c.startswith("semana") and pd.api.types.is_datetime64_any_dtype(pdf[c]):
            pdf[c] = pdf[c].dt.date
        elif pdf[c].dtype == object or pd.api.types.is_string_dtype(pdf[c]):
            col = pdf[c].astype(object)
            col[col.isna()] = None              # (where(..., None) en pandas vuelve a poner NaN)
            pdf[c] = col
    sdf = spark.createDataFrame(pdf)
    for c, tipo in sdf.dtypes:
        if tipo in ("double", "float"):
            sdf = sdf.withColumn(c, F.when(F.isnan(c), None).otherwise(F.col(c)))
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabla)
    print(f"Gold: {tabla}  ({spark.table(tabla).count()} filas)")

guardar(precio_semanal, T_PRECIO)
guardar(variacion_semanal, T_VARIACION)
guardar(features_modelo, T_FEATURES)

## 3. Modelo: regresión logística y árbol de decisión vs. línea base (MLflow)

In [0]:
import mlflow, mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

if not EXPERIMENTO:
    usuario = spark.sql("SELECT current_user()").first()[0]
    EXPERIMENTO = f"/Users/{usuario}/{CATALOGO}-gasolina"
mlflow.set_experiment(EXPERIMENTO)

entrena, prueba, semana_corte = separar_temporal(features_modelo)
print(f"Entrenamiento: {len(entrena)} filas ({entrena['semana_inicio'].min():%Y-%m-%d} a {entrena['semana_inicio'].max():%Y-%m-%d})")
print(f"Prueba:        {len(prueba)} filas ({prueba['semana_inicio'].min():%Y-%m-%d} a {prueba['semana_inicio'].max():%Y-%m-%d})")
print("Clases en entrenamiento:", entrena["direccion_siguiente"].value_counts().to_dict())

def metricas(real, pred):
    return {"accuracy": round(accuracy_score(real, pred), 4),
            "f1_macro": round(f1_score(real, pred, average="macro", labels=CLASES, zero_division=0), 4)}

candidatos = {
    "regresion_logistica": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
    "arbol_decision": DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, class_weight="balanced", random_state=42),
}
comunes = {"umbral_q": UMBRAL_Q, "semanas_prueba": SEMANAS_PRUEBA, "relleno_maga": RELLENO_MAGA,
           "semana_corte": str(semana_corte.date()), "features": ",".join(FEATURES),
           "filas_entrenamiento": len(entrena), "filas_prueba": len(prueba)}
resultados = []

# Línea base: "se repite la dirección de la semana anterior"
with mlflow.start_run(run_name="linea_base") as run:
    m = metricas(prueba["direccion_siguiente"], prueba["direccion"])
    mlflow.log_params({**comunes, "modelo": "linea_base"}); mlflow.log_metrics({f"test_{k}": v for k, v in m.items()})
    resultados.append({"modelo": "linea_base", **m, "run_id": run.info.run_id})

modelos_entrenados = {}
if entrena["direccion_siguiente"].nunique() < 2:
    print("⚠️  Solo hay una clase en entrenamiento: no se puede entrenar un clasificador; queda la línea base")
else:
    for nombre, modelo in candidatos.items():
        with mlflow.start_run(run_name=nombre) as run:
            modelo.fit(entrena[FEATURES], entrena["direccion_siguiente"])
            m = metricas(prueba["direccion_siguiente"], modelo.predict(prueba[FEATURES]))
            m_tr = metricas(entrena["direccion_siguiente"], modelo.predict(entrena[FEATURES]))
            mlflow.log_params({**comunes, "modelo": nombre, **{k: v for k, v in modelo.get_params().items()
                                                               if isinstance(v, (int, float, str, bool))}})
            mlflow.log_metrics({**{f"test_{k}": v for k, v in m.items()}, **{f"train_{k}": v for k, v in m_tr.items()}})
            mlflow.sklearn.log_model(modelo, "modelo", input_example=entrena[FEATURES].head(3),
                                     serialization_format="cloudpickle")
            resultados.append({"modelo": nombre, **m, "run_id": run.info.run_id})
            modelos_entrenados[nombre] = modelo

res = pd.DataFrame(resultados)
base = res[res["modelo"] == "linea_base"].iloc[0]
modelos = res[res["modelo"] != "linea_base"].sort_values(["f1_macro", "accuracy"], ascending=False)
elegido = modelos.iloc[0]["modelo"] if len(modelos) else "linea_base"
res["es_elegido"] = res["modelo"] == elegido
res["supera_linea_base"] = (res["f1_macro"] > base["f1_macro"]) | \
                           ((res["f1_macro"] == base["f1_macro"]) & (res["accuracy"] > base["accuracy"]))
print(res[["modelo", "accuracy", "f1_macro", "es_elegido", "supera_linea_base"]].to_string(index=False))
print(f"\nModelo elegido: {elegido}  ->  " +
      ("SUPERA a la línea base" if res.loc[res["es_elegido"], "supera_linea_base"].iloc[0] else "NO supera a la línea base"))

## 4. Predicciones (prueba fuera de muestra + semana siguiente)

In [0]:
def predecir(modelo, df):
    """Predicción + probabilidad de cada clase. Si no hay modelo (solo línea base), repite la dirección actual."""
    out = df[["combustible", "semana_inicio", "direccion", "direccion_siguiente"]].copy()
    if modelo is None:
        out["prediccion"] = df["direccion"].to_numpy()
        for c in CLASES:
            out[f"prob_{c}"] = (out["prediccion"] == c).astype(float)
    else:
        proba = modelo.predict_proba(df[FEATURES])
        clases = list(modelo.classes_)
        out["prediccion"] = modelo.predict(df[FEATURES])
        for c in CLASES:
            out[f"prob_{c}"] = proba[:, clases.index(c)].round(4) if c in clases else 0.0
    out["probabilidad"] = out[[f"prob_{c}" for c in CLASES]].max(axis=1)
    return out

modelo_eval = modelos_entrenados.get(elegido)

# Modelo final: el elegido, reentrenado con todas las semanas con etiqueta, para predecir la semana que viene
modelo_final, run_final = None, None
if modelo_eval is not None:
    from sklearn.base import clone
    todas = pd.concat([entrena, prueba])
    modelo_final = clone(modelo_eval).fit(todas[FEATURES], todas["direccion_siguiente"])
    with mlflow.start_run(run_name=f"final_{elegido}") as run:
        mlflow.log_params({**comunes, "modelo": elegido, "reentrenado_con": "entrenamiento+prueba",
                           "filas_finales": len(todas)})
        mlflow.sklearn.log_model(modelo_final, "modelo", input_example=todas[FEATURES].head(3),
                                 serialization_format="cloudpickle")
        run_final = run.info.run_id

ultima = features_modelo.sort_values("semana_inicio").groupby("combustible").tail(1)   # última semana con datos
ultima = ultima[ultima[FEATURES].notna().all(axis=1)]

partes = []
for conjunto, df, mod in (("entrenamiento", entrena, modelo_eval), ("prueba", prueba, modelo_eval),
                          ("futuro", ultima, modelo_final)):
    if len(df):
        p = predecir(mod, df)
        p["conjunto"] = conjunto
        partes.append(p)
pred = pd.concat(partes, ignore_index=True)
pred = pred.rename(columns={"semana_inicio": "semana_base", "direccion": "prediccion_linea_base",
                            "direccion_siguiente": "direccion_real"})
pred["semana"] = pred["semana_base"] + pd.Timedelta(days=7)          # semana que se predice (martes)
pred.loc[pred["conjunto"] == "futuro", "direccion_real"] = None
pred["acierto"] = np.where(pred["direccion_real"].isna(), None, pred["prediccion"] == pred["direccion_real"])
pred["modelo"] = elegido
pred["run_id"] = np.where(pred["conjunto"] == "futuro", run_final, res.loc[res["es_elegido"], "run_id"].iloc[0])
pred["generado_en"] = pd.Timestamp.now()
pred = pred[["combustible", "semana", "semana_base", "prediccion", "probabilidad", *[f"prob_{c}" for c in CLASES],
             "prediccion_linea_base", "direccion_real", "acierto", "conjunto", "modelo", "run_id", "generado_en"]]
guardar(pred, T_PREDICCIONES)

met = res.assign(conjunto="prueba", n_filas=len(prueba), n_semanas=prueba["semana_inicio"].nunique(),
                 semana_corte=semana_corte.date(), umbral_q=UMBRAL_Q, entrenado_en=pd.Timestamp.now())
guardar(met[["modelo", "conjunto", "accuracy", "f1_macro", "n_filas", "n_semanas", "es_elegido",
             "supera_linea_base", "semana_corte", "umbral_q", "run_id", "entrenado_en"]], T_METRICAS)

display(spark.table(T_PREDICCIONES).filter("conjunto = 'futuro'"))

## 5. Documentación de tablas y columnas (COMMENT)

In [0]:
DOC = {
    T_PRECIO: ("Precio por galón (Q) por semana martes-lunes y combustible. Base de /precios y /precios/actual.", {
        "combustible": "diesel, regular o super",
        "semana_inicio": "Martes en que inicia la semana (vigencia MEM martes a lunes)",
        "semana_fin": "Lunes en que termina la semana",
        "promedio": "Precio promedio Q/galón de la semana",
        "minimo": "Precio mínimo Q/galón leído en la semana",
        "maximo": "Precio máximo Q/galón leído en la semana",
        "n_lecturas": "Cantidad de lecturas de fotos usadas (0 si la semana no tiene fotos)",
        "fuente": "foto = promedio de fotos; maga_interpolado/maga_escalon = promedio mensual MAGA llevado a semana; arrastre = se repite el último precio conocido"}),
    T_VARIACION: ("Cambio de precio contra la semana anterior y etiqueta de dirección.", {
        "combustible": "diesel, regular o super",
        "semana_inicio": "Martes en que inicia la semana",
        "promedio": "Precio promedio Q/galón de la semana",
        "precio_anterior": "Precio promedio Q/galón de la semana anterior",
        "dif_q": "promedio - precio_anterior, en quetzales",
        "dif_pct": "dif_q como porcentaje de precio_anterior",
        "direccion": "sube si dif_q > umbral_q, baja si dif_q < -umbral_q, se_mantiene en otro caso",
        "umbral_q": "Umbral en Q usado para la etiqueta de dirección",
        "fuente": "Origen del precio de esta semana (ver gold.precio_semanal)",
        "fuente_anterior": "Origen del precio de la semana anterior"}),
    T_FEATURES: ("Variables para predecir la dirección del precio de la semana siguiente.", {
        "combustible": "diesel, regular o super",
        "semana_inicio": "Semana con la que se calculan las variables",
        "promedio": "Precio promedio Q/galón de la semana",
        "dif_1": "Cambio en Q contra la semana anterior",
        "dif_2": "Cambio en Q de hace 1 semana",
        "dif_3": "Cambio en Q de hace 2 semanas",
        "dif_4": "Cambio en Q de hace 3 semanas",
        "dif_pct_1": "Cambio % contra la semana anterior",
        "desvio_media_4": "Precio actual menos promedio móvil de 4 semanas",
        "dif_acum_4": "Cambio acumulado en Q de las últimas 4 semanas",
        "spread_super_diesel": "Precio súper menos diésel en la misma semana",
        "spread_regular_diesel": "Precio regular menos diésel en la misma semana",
        "dir_actual": "Dirección de esta semana codificada: -1 baja, 0 se_mantiene, 1 sube",
        "es_diesel": "1 si el combustible es diésel", "es_regular": "1 si el combustible es regular",
        "es_super": "1 si el combustible es súper",
        "media_movil_4": "Promedio móvil de 4 semanas del precio",
        "fuente": "Origen del precio de esta semana",
        "direccion": "Dirección de esta semana (sube/baja/se_mantiene)",
        "direccion_siguiente": "Objetivo: dirección de la semana siguiente",
        "apta_modelo": "true si la fila se usa para entrenar/evaluar (semanas observadas, sin arrastre, sin vacíos)"}),
    T_PREDICCIONES: ("Predicción de la dirección del precio por combustible y semana. Base de /prediccion.", {
        "combustible": "diesel, regular o super",
        "semana": "Martes de la semana que se predice",
        "semana_base": "Semana cuyos datos se usaron para predecir (la anterior a 'semana')",
        "prediccion": "Dirección predicha: sube, baja o se_mantiene",
        "probabilidad": "Probabilidad de la clase predicha",
        "prob_baja": "Probabilidad de baja", "prob_se_mantiene": "Probabilidad de se_mantiene",
        "prob_sube": "Probabilidad de sube",
        "prediccion_linea_base": "Predicción de la línea base (repite la dirección de semana_base)",
        "direccion_real": "Dirección observada; vacía si la semana aún no ocurre",
        "acierto": "true si prediccion = direccion_real",
        "conjunto": "entrenamiento (dentro de muestra), prueba (fuera de muestra) o futuro (semana siguiente)",
        "modelo": "Modelo elegido", "run_id": "Run de MLflow del modelo que hizo la predicción",
        "generado_en": "Fecha y hora de generación"}),
    T_METRICAS: ("Métricas en el conjunto de prueba (validación temporal) del modelo y la línea base. Base de /modelo/metricas.", {
        "modelo": "linea_base, regresion_logistica o arbol_decision",
        "conjunto": "Siempre prueba: semanas más recientes que no se usaron para entrenar",
        "accuracy": "Proporción de semanas en que se acertó la dirección",
        "f1_macro": "F1 promedio de las tres clases (no favorece a la clase más frecuente)",
        "n_filas": "Filas (combustible-semana) evaluadas", "n_semanas": "Semanas evaluadas",
        "es_elegido": "true para el modelo usado en gold.predicciones",
        "supera_linea_base": "true si su f1_macro (y luego accuracy) es mayor que el de la línea base",
        "semana_corte": "Primera semana del conjunto de prueba", "umbral_q": "Umbral sube/baja en Q",
        "run_id": "Run de MLflow", "entrenado_en": "Fecha y hora del entrenamiento"}),
}

def _q(texto):
    return texto.replace("'", "\\'")

for tabla, (comentario, cols) in DOC.items():
    spark.sql(f"COMMENT ON TABLE {tabla} IS '{_q(comentario)}'")
    existentes = set(spark.table(tabla).columns)
    for col, txt in cols.items():
        if col in existentes:
            spark.sql(f"ALTER TABLE {tabla} ALTER COLUMN {col} COMMENT '{_q(txt)}'")
print("Comentarios aplicados")

## 6. Pruebas automáticas (si algo falla, el Job falla)

In [0]:
ps, vs, pr = spark.table(T_PRECIO), spark.table(T_VARIACION), spark.table(T_PREDICCIONES)

assert ps.count() > 0, "precio_semanal vacía"
assert ps.groupBy("combustible", "semana_inicio").count().filter("count > 1").count() == 0, "semanas duplicadas"
assert ps.filter(F.dayofweek("semana_inicio") != 3).count() == 0, "hay semanas que no empiezan en martes"
assert ps.filter("minimo > promedio OR promedio > maximo").count() == 0, "mínimo/promedio/máximo inconsistentes"
assert ps.filter("promedio NOT BETWEEN 5 AND 100").count() == 0, "precios fuera de rango"
assert vs.filter(f"(abs(dif_q) <= {UMBRAL_Q} AND direccion <> 'se_mantiene') OR "
                 f"(dif_q > {UMBRAL_Q} AND direccion <> 'sube') OR (dif_q < -{UMBRAL_Q} AND direccion <> 'baja')").count() == 0, \
    "etiqueta de dirección inconsistente con el umbral"
assert entrena["semana_inicio"].max() < prueba["semana_inicio"].min(), "la prueba no es posterior al entrenamiento"
fut = pr.filter("conjunto = 'futuro'")
assert fut.count() == len(OFICIALES) and fut.select("combustible").distinct().count() == len(OFICIALES), \
    "debe haber una predicción futura por combustible"
assert pr.groupBy("combustible", "semana", "conjunto").count().filter("count > 1").count() == 0, "predicciones duplicadas"
assert spark.table(T_METRICAS).filter("es_elegido").count() == 1, "debe haber un solo modelo elegido"
print("✅ Todas las pruebas pasaron")

## 7. Revisión visual

In [0]:
display(spark.table(T_METRICAS).select("modelo", "accuracy", "f1_macro", "n_semanas", "es_elegido", "supera_linea_base"))
display(spark.table(T_PRECIO).groupBy("combustible", "fuente")
        .agg(F.min("semana_inicio").alias("desde"), F.max("semana_inicio").alias("hasta"), F.count("*").alias("semanas"))
        .orderBy("combustible", "fuente"))
display(spark.table(T_PREDICCIONES).filter("conjunto = 'prueba'")
        .groupBy("combustible").agg(F.avg(F.col("acierto").cast("int")).alias("exactitud_modelo"),
                                    F.avg((F.col("prediccion_linea_base") == F.col("direccion_real")).cast("int"))
                                    .alias("exactitud_linea_base")))

In [0]:
import matplotlib.pyplot as plt
COLORES = {"diesel": "#333333", "regular": "#E0B000", "super": "#2E8B57"}
d = precio_semanal[precio_semanal["semana_inicio"] >= precio_semanal["semana_inicio"].max() - pd.Timedelta(weeks=104)]
fig, ax = plt.subplots(figsize=(11, 4.5))
for c in OFICIALES:
    s = d[d["combustible"] == c]
    ax.plot(s["semana_inicio"], s["promedio"], color=COLORES[c], lw=1.3, label=c)
    f = s[s["fuente"] == "foto"]
    ax.scatter(f["semana_inicio"], f["promedio"], color=COLORES[c], s=18, zorder=3)
    if len(prueba):
        ax.axvline(semana_corte, color="gray", ls=":", lw=1)
ax.set_title("Precio semanal (últimos 2 años) · puntos = semanas con fotos · línea punteada = inicio de prueba")
ax.set_ylabel("Q/galón"); ax.grid(alpha=0.3); ax.legend(frameon=False)
plt.tight_layout(); plt.show()